In [1]:
!pip install pandas openpyxl faker xlsxwriter


You should consider upgrading via the 'C:\Users\kardi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
import pandas as pd
import random
from faker import Faker

from openpyxl import load_workbook

# Excel formatting
import xlsxwriter

In [3]:
# -------------------------
# INITIALIZE FAKER
# -------------------------

fake = Faker("en_IN")

In [4]:
# =========================================================
# CONFIGURATION
# =========================================================
NUM_RECORDS = 500

FIELD_WEIGHTS = {
    "Name 1": 20,
    "Tax Number": 18,
    "Street": 12,
    "Industry Sector": 5,
    "Email": 7
}

In [5]:
# =========================================================
# HELPER FUNCTION
# Randomly make some fields blank to simulate poor SAP data
# =========================================================

def maybe_missing(value, probability=0.15):

    if random.random() < probability:
        return None

    return value


In [6]:
# =========================================================
# GENERATE DUMMY BUSINESS PARTNER DATA
# =========================================================

records = []

for i in range(NUM_RECORDS):

    bp_number = f"BP{100000 + i}"

    record = {

        "Business Partner": bp_number,

        "Name 1": maybe_missing(
            fake.company(),
            0.05
        ),

        "Street": maybe_missing(
            fake.street_address(),
            0.18
        ),

        "City": fake.city(),

        "Postal Code": fake.postcode(),

        "Country/Region": "IN",

        "Tax Number": maybe_missing(
            f"{random.randint(10,99)}ABCDE{random.randint(1000,9999)}F1Z5",
            0.22
        ),

        "Industry Sector": maybe_missing(
            random.choice([
                "Manufacturing",
                "Retail",
                "IT",
                "Healthcare",
                "Logistics"
            ]),
            0.10
        ),

        "Email": maybe_missing(
            fake.email(),
            0.12
        ),

        "Created By": random.choice([
            "SAPUSER1",
            "SAPUSER2",
            "ADMIN",
            "MDM_USER"
        ])
    }

    records.append(record)

In [7]:
# =========================================================
# CREATE DATAFRAME
# =========================================================

df = pd.DataFrame(records)

print("Dummy SAP BP data generated successfully.")
print(df.head())


Dummy SAP BP data generated successfully.
  Business Partner                     Name 1                 Street  \
0         BP100000       Lata, Johal and Kota         H.No. 32\nSant   
1         BP100001     Ghosh, Dhar and Bansal      26/135\nMane Path   
2         BP100002    Ratta, Rao and Krishnan        16, Mittal Marg   
3         BP100003  Sheth, Sharaf and Kashyap                   None   
4         BP100004         Venkataraman-Samra  H.No. 416\nBobal Path   

          City Postal Code Country/Region       Tax Number Industry Sector  \
0        Mango      735239             IN  94ABCDE4610F1Z5          Retail   
1        Dewas      140628             IN  88ABCDE9637F1Z5   Manufacturing   
2     Gopalpur      726375             IN  39ABCDE7803F1Z5            None   
3     Agartala      142963             IN  38ABCDE8001F1Z5              IT   
4  Bhubaneswar      987696             IN             None       Logistics   

                         Email Created By  
0           

In [8]:
# =========================================================
# VALIDATE EACH BUSINESS PARTNER RECORD
# Identify missing critical fields
# =========================================================

def validate_row(row):

    issues = []

    # Check missing company name
    if pd.isna(row.get("Name 1")):
        issues.append("Missing Name")

    # Check missing tax number
    if pd.isna(row.get("Tax Number")):
        issues.append("Missing Tax Number")

    # Check missing street address
    if pd.isna(row.get("Street")):
        issues.append("Missing Address")

    # FIXED BUG:
    # Check missing industry sector
    if pd.isna(row.get("Industry Sector")):
        issues.append("Missing Industry")

    # Check missing email
    if pd.isna(row.get("Email")):
        issues.append("Missing Email")

    return issues


In [9]:

# =========================================================
# APPLY VALIDATION LOGIC
# =========================================================

df["Issues"] = df.apply(validate_row, axis=1)

In [10]:
# =========================================================
# SCORE EACH RECORD BASED ON DATA QUALITY
# =========================================================

def calculate_score(row):

    score = 100

    # Deduct points for missing company name
    if "Missing Name" in row["Issues"]:
        score -= FIELD_WEIGHTS["Name 1"]

    # Deduct points for missing tax number
    if "Missing Tax Number" in row["Issues"]:
        score -= FIELD_WEIGHTS["Tax Number"]

    # Deduct points for missing address
    if "Missing Address" in row["Issues"]:
        score -= FIELD_WEIGHTS["Street"]

    # Deduct points for missing industry
    if "Missing Industry" in row["Issues"]:
        score -= FIELD_WEIGHTS["Industry Sector"]

    # Deduct points for missing email
    if "Missing Email" in row["Issues"]:
        score -= FIELD_WEIGHTS["Email"]

    return max(score, 0)

In [11]:
# =========================================================
# APPLY SCORE CALCULATION
# =========================================================

df["Score"] = df.apply(calculate_score, axis=1)

In [12]:
# =========================================================
# CLASSIFY RECORD HEALTH STATUS
# =========================================================

def classify_status(score):

    if score < 50:
        return "CRITICAL"

    elif score < 70:
        return "POOR"

    elif score < 90:
        return "REVIEW"

    else:
        return "HEALTHY"

In [13]:
# =========================================================
# APPLY STATUS CLASSIFICATION
# =========================================================

df["Status"] = df["Score"].apply(classify_status)

In [15]:
# =========================================================
# IDENTIFY WHICH IMPORTANT FIELDS ARE MISSING
# =========================================================

important_fields = [
    "Name 1",
    "Tax Number",
    "Street",
    "Industry Sector",
    "Email"
]

def get_missing_fields(row):

    missing = []

    for field in important_fields:

        if pd.isna(row.get(field)):
            missing.append(field)

    return ", ".join(missing)

In [16]:
# =========================================================
# APPLY MISSING FIELD IDENTIFICATION
# =========================================================

df["Missing Fields"] = df.apply(
    get_missing_fields,
    axis=1
)

In [17]:
# =========================================================
# COUNT NUMBER OF MISSING IMPORTANT FIELDS
# =========================================================

df["Missing Count"] = df["Missing Fields"].apply(

    lambda x: 0 if x == "" else len(x.split(", "))
)


In [18]:
# =========================================================
# SORT RECORDS BY CLEANUP PRIORITY
# Worst records appear first
# =========================================================

df_sorted = df.sort_values(

    by=["Missing Count", "Score"],

    ascending=[False, True]
)


In [19]:
# =========================================================
# DASHBOARD KPI CALCULATIONS
# =========================================================

total_records = len(df)

average_score = round(df["Score"].mean(), 2)

healthy_count = (
    df["Status"] == "HEALTHY"
).sum()

review_count = (
    df["Status"] == "REVIEW"
).sum()

poor_count = (
    df["Status"] == "POOR"
).sum()

critical_count = (
    df["Status"] == "CRITICAL"
).sum()

In [20]:
# =========================================================
# CREATE STATUS BREAKDOWN TABLE
# =========================================================

status_summary = df["Status"].value_counts().reset_index()

status_summary.columns = [
    "Status",
    "Count"
]



In [21]:
# =========================================================
# ROOT CAUSE ANALYSIS
# Count missing values by field
# =========================================================

missing_analysis = (
    df[important_fields]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

In [22]:
# =========================================================
# CREATE CLEANUP TABLE
# =========================================================

cleanup_columns = [
    "Business Partner",
    "Score",
    "Status",
    "Missing Fields"
]

cleanup_df = df_sorted[cleanup_columns]


In [23]:
# =========================================================
# EXPORT FINAL DASHBOARD REPORT
# =========================================================

output_file = "../data/BP_Quality_Dashboard_Report.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="xlsxwriter"
) as writer:
 # =====================================================
    # WRITE DASHBOARD SHEET
    # =====================================================

    dashboard_df = pd.DataFrame({

        "Metric": [
            "Total Records",
            "Average Score",
            "Healthy Records",
            "Review Records",
            "Poor Records",
            "Critical Records"
        ],

        "Value": [
            total_records,
            average_score,
            healthy_count,
            review_count,
            poor_count,
            critical_count
        ]
    })

    dashboard_df.to_excel(
        writer,
        sheet_name="Dashboard",
        startrow=2,
        index=False
    )

    status_summary.to_excel(
        writer,
        sheet_name="Dashboard",
        startrow=12,
        index=False
    )
# =====================================================
    # WRITE CLEANUP SHEET
    # ===================================================

    cleanup_df.to_excel(
        writer,
        sheet_name="Cleanup List",
        index=False
    )
  # =====================================================
    # WRITE ROOT CAUSE ANALYSIS SHEET
    # =====================================================
    missing_analysis_df = (
        missing_analysis
        .reset_index()
    )

    missing_analysis_df.columns = [
        "Field",
        "Missing Count"
    ]

    missing_analysis_df.to_excel(
        writer,
        sheet_name="Root Cause Analysis",
        index=False
    )
    # =====================================================
    # ACCESS WORKBOOK & WORKSHEETS
    # =====================================================
    workbook = writer.book

    dashboard_ws = writer.sheets["Dashboard"]

    cleanup_ws = writer.sheets["Cleanup List"]

    rootcause_ws = writer.sheets["Root Cause Analysis"]
      
       # =====================================================
    # DEFINE FORMATS
    # =====================================================
     # Dashboard title format
    title_format = workbook.add_format({

        "bold": True,
        "font_size": 18,
        "bg_color": "#1F4E78",
        "font_color": "#FFFFFF",
        "align": "center",
        "valign": "vcenter"
    })

    # Header format
    header_format = workbook.add_format({

        "bold": True,
        "bg_color": "#1F3864",
        "font_color": "#FFFFFF",
        "border": 1
    })
    # Critical row format
    critical_format = workbook.add_format({

        "bg_color": "#FFC7CE"
    })
     # Poor row format
    poor_format = workbook.add_format({

        "bg_color": "#FFEB9C"
    })
    # REVIEW color added (orange)
    review_format = workbook.add_format({

        "bg_color": "#F4B183"
    })
    # Healthy row format
    healthy_format = workbook.add_format({

        "bg_color": "#C6EFCE"
    })

  # =====================================================
    # FORMAT DASHBOARD SHEET
    # =====================================================

    dashboard_ws.merge_range(

        "A1:D1",

        "BUSINESS PARTNER DATA QUALITY DASHBOARD",

        title_format
    )

    # Apply header styling
    dashboard_ws.write("A3", "Metric", header_format)
    dashboard_ws.write("B3", "Value", header_format)

    dashboard_ws.write("A13", "Status", header_format)
    dashboard_ws.write("B13", "Count", header_format)

    dashboard_ws.set_column("A:A", 30)
    dashboard_ws.set_column("B:B", 18)


    # =====================================================
    # FORMAT CLEANUP SHEET
    # =====================================================
    for col_num, value in enumerate(cleanup_df.columns.values):

        cleanup_ws.write(
            0,
            col_num,
            value,
            header_format
        )

    cleanup_ws.set_column("A:A", 20)
    cleanup_ws.set_column("B:B", 12)
    cleanup_ws.set_column("C:C", 15)
    cleanup_ws.set_column("D:D", 50)

    # Freeze header row
    cleanup_ws.freeze_panes(1, 0)

    # =====================================================
    # APPLY STATUS COLOR FORMATTING
    # =====================================================

    for row_num in range(1, len(cleanup_df) + 1):

        status = cleanup_df.iloc[row_num - 1]["Status"]

        if status == "CRITICAL":

            cleanup_ws.set_row(
                row_num,
                cell_format=critical_format
            )

        elif status == "POOR":

            cleanup_ws.set_row(
                row_num,
                cell_format=poor_format
            )

        elif status == "REVIEW":
            cleanup_ws.set_row(
                row_num,
                cell_format=review_format
            )

        elif status == "HEALTHY":

            cleanup_ws.set_row(
                row_num,
                cell_format=healthy_format
            )
     # =====================================================
    # FORMAT ROOT CAUSE ANALYSIS SHEET
    # =====================================================
    for col_num, value in enumerate(
        missing_analysis_df.columns.values
    ):

        rootcause_ws.write(
            0,
            col_num,
            value,
            header_format
        )

    rootcause_ws.set_column("A:A", 30)
    rootcause_ws.set_column("B:B", 20)

    rootcause_ws.freeze_panes(1, 0)

      # =====================================================
    # CREATE PIE CHART
    # =====================================================

    
    chart = workbook.add_chart({

        "type": "pie"
    })

    chart.add_series({

        "name": "BP Health Distribution",

        "categories": [
            "Dashboard",
            13,
            0,
            16,
            0
        ],

        "values": [
            "Dashboard",
            13,
            1,
            16,
            1
        ]
    })

    chart.set_title({

        "name": "BP Health Status Breakdown"
    })

    dashboard_ws.insert_chart("D3", chart)

    # =========================================================
# FINAL MESSAGE
# =========================================================
print("=" * 60)
print("BUSINESS PARTNER DATA QUALITY DASHBOARD GENERATED")
print("=" * 60)

print(f"\nOutput File: {output_file}")

print("\nProject completed successfully.")






OSError: Cannot save file into a non-existent directory: '..\data'

In [24]:
# =========================================================
# EXPORT FINAL DASHBOARD REPORT
# =========================================================

output_file = "../data/BP_Quality_Dashboard_Report.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="xlsxwriter"
) as writer:

    # =====================================================
    # WRITE DASHBOARD SHEET
    # =====================================================

    dashboard_df = pd.DataFrame({

        "Metric": [
            "Total Records",
            "Average Score",
            "Healthy Records",
            "Review Records",
            "Poor Records",
            "Critical Records"
        ],

        "Value": [
            total_records,
            average_score,
            healthy_count,
            review_count,
            poor_count,
            critical_count
        ]
    })

    dashboard_df.to_excel(
        writer,
        sheet_name="Dashboard",
        startrow=2,
        index=False
    )

    status_summary.to_excel(
        writer,
        sheet_name="Dashboard",
        startrow=12,
        index=False
    )

    # =====================================================
    # WRITE CLEANUP SHEET
    # =====================================================

    cleanup_df.to_excel(
        writer,
        sheet_name="Cleanup List",
        index=False
    )

    # =====================================================
    # WRITE ROOT CAUSE ANALYSIS SHEET
    # =====================================================

    missing_analysis_df = (
        missing_analysis
        .reset_index()
    )

    missing_analysis_df.columns = [
        "Field",
        "Missing Count"
    ]

    missing_analysis_df.to_excel(
        writer,
        sheet_name="Root Cause Analysis",
        index=False
    )

    # =====================================================
    # ACCESS WORKBOOK & WORKSHEETS
    # =====================================================

    workbook = writer.book

    dashboard_ws = writer.sheets["Dashboard"]

    cleanup_ws = writer.sheets["Cleanup List"]

    rootcause_ws = writer.sheets["Root Cause Analysis"]

    # =====================================================
    # DEFINE FORMATS
    # =====================================================

    # Dashboard title format
    title_format = workbook.add_format({

        "bold": True,
        "font_size": 18,
        "bg_color": "#1F4E78",
        "font_color": "#FFFFFF",
        "align": "center",
        "valign": "vcenter"
    })

    # Header format
    header_format = workbook.add_format({

        "bold": True,
        "bg_color": "#1F3864",
        "font_color": "#FFFFFF",
        "border": 1
    })

    # Critical row format
    critical_format = workbook.add_format({

        "bg_color": "#FFC7CE"
    })

    # Poor row format
    poor_format = workbook.add_format({

        "bg_color": "#FFEB9C"
    })

    # Review row format
    review_format = workbook.add_format({

        "bg_color": "#F4B183"
    })

    # Healthy row format
    healthy_format = workbook.add_format({

        "bg_color": "#C6EFCE"
    })

    # =====================================================
    # FORMAT DASHBOARD SHEET
    # =====================================================

    dashboard_ws.merge_range(

        "A1:D1",

        "BUSINESS PARTNER DATA QUALITY DASHBOARD",

        title_format
    )

    # Apply dashboard headers
    dashboard_ws.write("A3", "Metric", header_format)
    dashboard_ws.write("B3", "Value", header_format)

    dashboard_ws.write("A13", "Status", header_format)
    dashboard_ws.write("B13", "Count", header_format)

    dashboard_ws.set_column("A:A", 30)
    dashboard_ws.set_column("B:B", 18)

    # =====================================================
    # FORMAT CLEANUP SHEET
    # =====================================================

    for col_num, value in enumerate(cleanup_df.columns.values):

        cleanup_ws.write(
            0,
            col_num,
            value,
            header_format
        )

    cleanup_ws.set_column("A:A", 20)
    cleanup_ws.set_column("B:B", 12)
    cleanup_ws.set_column("C:C", 15)
    cleanup_ws.set_column("D:D", 50)

    # Freeze top row
    cleanup_ws.freeze_panes(1, 0)

    # =====================================================
    # APPLY STATUS COLOR FORMATTING
    # =====================================================

    for row_num in range(1, len(cleanup_df) + 1):

        status = cleanup_df.iloc[row_num - 1]["Status"]

        if status == "CRITICAL":

            cleanup_ws.set_row(
                row_num,
                cell_format=critical_format
            )

        elif status == "POOR":

            cleanup_ws.set_row(
                row_num,
                cell_format=poor_format
            )

        elif status == "REVIEW":

            cleanup_ws.set_row(
                row_num,
                cell_format=review_format
            )

        elif status == "HEALTHY":

            cleanup_ws.set_row(
                row_num,
                cell_format=healthy_format
            )

    # =====================================================
    # FORMAT ROOT CAUSE ANALYSIS SHEET
    # =====================================================

    for col_num, value in enumerate(
        missing_analysis_df.columns.values
    ):

        rootcause_ws.write(
            0,
            col_num,
            value,
            header_format
        )

    rootcause_ws.set_column("A:A", 30)
    rootcause_ws.set_column("B:B", 20)

    rootcause_ws.freeze_panes(1, 0)

    # =====================================================
    # CREATE PIE CHART
    # =====================================================

    chart = workbook.add_chart({

        "type": "pie"
    })

    chart.add_series({

        "name": "BP Health Distribution",

        "categories": [
            "Dashboard",
            13,
            0,
            16,
            0
        ],

        "values": [
            "Dashboard",
            13,
            1,
            16,
            1
        ]
    })

    chart.set_title({

        "name": "BP Health Status Breakdown"
    })

    dashboard_ws.insert_chart("D3", chart)

# =========================================================
# FINAL MESSAGE
# =========================================================

print("=" * 60)
print("BUSINESS PARTNER DATA QUALITY DASHBOARD GENERATED")
print("=" * 60)

print(f"\nOutput File: {output_file}")

print("\nProject completed successfully.")

OSError: Cannot save file into a non-existent directory: '..\data'